In [ ]:
import numpy as np
import pandas as pd
import math

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.multioutput import MultiOutputRegressor

from xgboost import XGBRegressor
from scipy.optimize import curve_fit

In [48]:

DATA_PATH = "data/llm_pilot_data/raw_data/per_model/data_bigcode_starcoder.csv"  # adjust if needed

df_all = pd.read_csv(DATA_PATH)
df_all.columns = df_all.columns.str.strip()
print("Raw shape:", df_all.shape)
print("Columns:", df_all.columns)

Raw shape: (134772, 53)
Columns: Index(['Workload_smpnum', 'Workload_reqnum', 'AI_model', 'Workload_num_users',
       'Workload_requests', 'Target_latency_ms', 'records', 'NonAI_n_gpus',
       'NonAI_gpu_type', 'Target_experiment_duration_s',
       'Workload_n_input_tokens', 'Workload_n_output_tokens',
       'Target_latency_ms_per_token', 'Target_timestamps_per_token',
       'NonAI_gpu', 'AI_model_is_flash_attention', 'AI_model_n_parameters',
       'AI_model_is_encoder_decoder', 'AI_model_type', 'AI_model_n_positions',
       'AI_model_n_heads', 'AI_model_n_layers',
       'AI_model_relative_attention_max_distance',
       'AI_model_relative_attention_n_buckets', 'AI_model_torch_dtype',
       'AI_model_vocabulary_size', 'NonAI_gpu_architecture',
       'NonAI_gpu_n_tensor_cores', 'NonAI_gpu_n_rt_cores',
       'NonAI_gpu_n_cuda_cores', 'NonAI_gpu_n_sms', 'NonAI_gpu_n_rops',
       'NonAI_gpu_n_tmus', 'NonAI_gpu_tflops_cuda_mixed',
       'NonAI_gpu_tflops_cuda_fp32', 'NonAI_gpu_

In [49]:
# Column mapping
COL_II  = "Workload_n_input_tokens"   # ii
COL_OO  = "Workload_n_output_tokens"  # oo
COL_BB  = "Workload_num_users"        # bb (concurrency)
COL_THROUGHPUT = "Target_throughput_tokens_per_sec"         # throughput target

In [22]:
df_all.columns.to_list()

['Workload_smpnum',
 'Workload_reqnum',
 'AI_model',
 'Workload_num_users',
 'Workload_requests',
 'Target_latency_ms',
 'records',
 'NonAI_n_gpus',
 'NonAI_gpu_type',
 'Target_experiment_duration_s',
 'Workload_n_input_tokens',
 'Workload_n_output_tokens',
 'Target_latency_ms_per_token',
 'Target_timestamps_per_token',
 'NonAI_gpu',
 'AI_model_is_flash_attention',
 'AI_model_n_parameters',
 'AI_model_is_encoder_decoder',
 'AI_model_type',
 'AI_model_n_positions',
 'AI_model_n_heads',
 'AI_model_n_layers',
 'AI_model_relative_attention_max_distance',
 'AI_model_relative_attention_n_buckets',
 'AI_model_torch_dtype',
 'AI_model_vocabulary_size',
 'NonAI_gpu_architecture',
 'NonAI_gpu_n_tensor_cores',
 'NonAI_gpu_n_rt_cores',
 'NonAI_gpu_n_cuda_cores',
 'NonAI_gpu_n_sms',
 'NonAI_gpu_n_rops',
 'NonAI_gpu_n_tmus',
 'NonAI_gpu_tflops_cuda_mixed',
 'NonAI_gpu_tflops_cuda_fp32',
 'NonAI_gpu_tflops_cuda_fp64',
 'NonAI_gpu_tflops_tc_bf16',
 'NonAI_gpu_tflops_tc_fp16',
 'NonAI_gpu_tflops_tc_fp3

In [50]:
df_all = pd.read_csv(DATA_PATH)
print("Raw shape:", df_all.shape)

# Keep only rows with required columns and positive values
needed_cols = [COL_II, COL_OO, COL_BB, COL_THROUGHPUT]
df_all = df_all[needed_cols].dropna().copy()

df_all = df_all[
    (df_all[COL_II]  > 0) &
    (df_all[COL_OO]  > 0) &
    (df_all[COL_BB]  > 0) &
    (df_all[COL_THROUGHPUT] > 0)
].copy()

df_all = df_all.astype(float)

print("After filtering:", df_all.shape)
df_all.head()

Raw shape: (134772, 53)
After filtering: (134772, 4)


,Workload_n_input_tokens,Workload_n_output_tokens,Workload_num_users,Target_throughput_tokens_per_sec
0,16.0,50.0,64.0,11.950025
1,13.0,1.0,64.0,8.793964
2,16.0,50.0,64.0,10.658913
3,13.0,1.0,64.0,8.793964
4,85.0,20.0,64.0,43.245452


In [40]:
def exp_throughput(bb, a, b, c):
    """
    Saturating exponential throughput model:
        throughput(bb) = a + c * (1 - exp(-b * bb))

    a: base latency (low concurrency)
    c: additional latency due to congestion
    b: sensitivity to concurrency
    """
    return a + c * (1.0 - np.exp(-b * bb))

In [41]:
# ============================================================
# 3. Fit (a, b, c) per (ii, oo) group
# ============================================================

def fit_throughput_group(bb_vals, thr_vals):
    """
    Fit (a, b, c) for a single (ii, oo) group using a saturating exponential:
        thr(bb) = a + c * (1 - exp(-b * bb))

    Robust init + bounds:
      a >= 0, b >= 0, c >= 0
    """
    bb = np.asarray(bb_vals, dtype=float)
    thr = np.asarray(thr_vals, dtype=float)

    # Basic sanity: non-negative throughput
    thr = np.maximum(thr, 0.0)

    if len(np.unique(bb)) < 2:
        # fallback: constant-ish throughput
        a0 = float(np.percentile(thr, 10))
        c0 = float(max(np.percentile(thr, 90) - a0, 1e-3))
        b0 = 0.01
        return (a0, b0, c0)

    thr_p10, thr_p90 = np.percentile(thr, [10, 90])
    bb_p10, bb_p90   = np.percentile(bb,  [10, 90])

    eps = 1e-3
    bb_p90 = max(bb_p90, bb_p10 + eps)

    a0 = float(max(thr_p10, 0.0))
    c0 = float(max(thr_p90 - thr_p10, 1e-3))
    b0 = float(1.0 / max(bb_p90 - bb_p10, 1e-3))

    p0 = (a0, b0, c0)
    bounds = ([0.0, 0.0, 0.0], [np.inf, np.inf, np.inf])

    try:
        popt, _ = curve_fit(
            exp_throughput,
            bb,
            thr,
            p0=p0,
            bounds=bounds,
            maxfev=5000,
        )
        return tuple(map(float, popt))
    except Exception:
        return (a0, b0, c0)


In [42]:
# ============================================================
# 4. Build parameter DB + XGBoost training table
# ============================================================

import pandas as pd

def build_throughput_db_and_training_params(df_train):
    """
    For each (ii, oo) group in df_train:
      - fit (a, b, c) of the throughput saturating exponential
      - store in param_db[(ii, oo)]
      - create a row in T_df with features + targets (a, b, c)
    """
    groups = df_train.groupby([COL_II, COL_OO], dropna=False)

    param_db = {}
    records = []

    for (ii, oo), g in groups:
        ii_f, oo_f = float(ii), float(oo)

        a_hat, b_hat, c_hat = fit_throughput_group(
            g[COL_BB].values,
            g[COL_THROUGHPUT].values,
        )

        key = (ii_f, oo_f)
        param_db[key] = (a_hat, b_hat, c_hat)

        feat = make_param_features(ii_f, oo_f).ravel()
        # feat order: [ii, oo, logii, logoo, logratio, ii_oo_ratio, ii_ii_ratio]
        records.append({
            "ii": feat[0],
            "oo": feat[1],
            "logii": feat[2],
            "logoo": feat[3],
            "logratio": feat[4],
            "ii_oo_ratio": feat[5],
            "ii_ii_ratio": feat[6],
            "a": a_hat,
            "b": b_hat,
            "c": c_hat,
        })

    T_df = pd.DataFrame(records)
    return param_db, T_df


In [35]:
# ============================================================
# 5. Train XGBoost parameter regressor (multi-output)
# ============================================================

def train_param_regressor(T_df):
    """
    Train MultiOutput XGBoost to predict (a, b, c)
    from (ii, oo)-derived features.
    """
    if T_df.empty:
        return None

    feature_cols = ["ii", "oo", "logii", "logoo",
                    "logratio", "ii_oo_ratio", "ii_ii_ratio"]
    target_cols = ["a", "b", "c"]

    X = T_df[feature_cols].values
    y = T_df[target_cols].values

    base_xgb = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=42,
    )

    model = MultiOutputRegressor(base_xgb)
    model.fit(X, y)
    return model


In [43]:
def ala_predict_throughput(df_rows, param_db, param_regressor, clip_nonneg: bool = True):
    preds = []

    for _, row in df_rows.iterrows():
        ii = float(row[COL_II])
        oo = float(row[COL_OO])
        bb = float(row[COL_BB])

        key = (ii, oo)

        if key in param_db:
            a_hat, b_hat, c_hat = param_db[key]
        else:
            X_feat = make_param_features(ii, oo)  # shape (1,7)
            a_hat, b_hat, c_hat = param_regressor.predict(X_feat)[0]

            # Safety: keep parameters in a sane domain
            a_hat = float(max(a_hat, 0.0))
            b_hat = float(max(b_hat, 0.0))
            c_hat = float(max(c_hat, 0.0))

        y = float(exp_throughput(bb, a_hat, b_hat, c_hat))
        if clip_nonneg:
            y = max(y, 0.0)

        preds.append(y)

    return np.asarray(preds, dtype=float)


In [44]:
def make_param_features(ii, oo):
    ii = float(ii)
    oo = float(oo)

    logii      = np.log1p(ii)
    logoo      = np.log1p(oo)
    logratio   = np.log1p(ii / (oo + 1e-6))
    ii_oo_ratio = ii / (oo + 1.0)
    ii_ii_ratio = ii / (ii + 1.0)

    return np.array([[ii, oo, logii, logoo, logratio,
                      ii_oo_ratio, ii_ii_ratio]], dtype=float)


In [45]:
# ============================================================
# 7. Metrics function (R2, RMSE, MAE, MAPE, MdAPE)
# ============================================================

def compute_metrics(y_true, y_pred, eps: float = 1e-8) -> dict:
    """
    Regression metrics:
      - R2
      - MAE
      - RMSE
      - MRE (%) = mean(|y_true - y_pred| / max(|y_true|, eps)) * 100
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))

    denom = np.maximum(np.abs(y_true), eps)
    mre = float(np.mean(np.abs(y_true - y_pred) / denom) * 100.0)

    return {
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse,
        "MRE": mre,
    }

In [46]:
import numpy as np
from sklearn.model_selection import train_test_split

def run_ala_split80_throughput(
    df,
    random_state: int = 42,
    test_size: float = 0.20,
    clip: bool = True,
):
    """
    Run the ALA-style pipeline with a single 80/20 split on throughput target.
    """

    # --------- split ----------
    df_train, df_test = train_test_split(
        df,
        test_size=test_size,
        shuffle=True,
        random_state=random_state,
    )

    # --------- optional clipping range (robustness) ----------
    # Throughput can have heavier tails; use percentiles.
    y_all = df[COL_THROUGHPUT].values.astype(float)
    y_p01 = np.percentile(y_all, 1)
    y_p99 = np.percentile(y_all, 99)

    # --------- train ----------
    # NOTE: replace these with your actual throughput equivalents.
    param_db, T_df = build_throughput_db_and_training_params(df_train)
    param_regressor = train_param_regressor(T_df)

    # --------- predict ----------
    y_true = df_test[COL_THROUGHPUT].values.astype(float)
    y_pred_raw = ala_predict_throughput(df_test, param_db, param_regressor)

    # --------- clip (recommended) ----------
    if clip:
        # lower bound at 0 (throughput cannot be negative), upper bound loosened from p99
        y_pred = np.clip(y_pred_raw, 0.0, y_p99 * 2.0)
    else:
        y_pred = np.asarray(y_pred_raw, dtype=float)

    # --------- metrics ----------
    m = compute_metrics(y_true, y_pred)

    print(
        f"[ALA throughput split80/20] "
        f"R2={m['R2']:.4f}, RMSE={m['RMSE']:.4f}, "
        f"MAE={m['MAE']:.4f}, MRE={m['MRE']:.2f}% "
        f"(train={len(df_train)}, test={len(df_test)})"
    )

    summary = {
        "target": COL_THROUGHPUT,
        "cv": "split80_20",
        "random_state": random_state,
        "test_size": test_size,
        "n_train": int(len(df_train)),
        "n_test": int(len(df_test)),
        **m,
    }

    return {"metrics": m, "summary": summary, "y_true": y_true, "y_pred": y_pred}


In [52]:
import numpy as np

# Choose 5 distinct seeds (same strategy you used elsewhere)
SEEDS = [42, 1042, 2042, 3042, 4042]

all_metrics = []

for i, seed in enumerate(SEEDS, start=1):
    print(f"\n=== ALA throughput split80 | run {i}/5 | seed={seed} ===")

    result = run_ala_split80_throughput(
        df=df_all,
        random_state=seed,
        test_size=0.20,
    )

    m = result["metrics"]
    all_metrics.append(m)

    print(
        f"R2={m['R2']:.4f}, "
        f"RMSE={m['RMSE']:.4f}, "
        f"MAE={m['MAE']:.4f}, "
        f"MRE={m['MRE']:.2f}%"
    )



=== ALA throughput split80 | run 1/5 | seed=42 ===
[ALA throughput split80/20] R2=0.4823, RMSE=432.7670, MAE=161.2694, MRE=167.03% (train=107817, test=26955)
R2=0.4823, RMSE=432.7670, MAE=161.2694, MRE=167.03%

=== ALA throughput split80 | run 2/5 | seed=1042 ===
[ALA throughput split80/20] R2=0.4711, RMSE=445.3028, MAE=160.1747, MRE=171.07% (train=107817, test=26955)
R2=0.4711, RMSE=445.3028, MAE=160.1747, MRE=171.07%

=== ALA throughput split80 | run 3/5 | seed=2042 ===
[ALA throughput split80/20] R2=0.4914, RMSE=429.6987, MAE=161.5488, MRE=165.32% (train=107817, test=26955)
R2=0.4914, RMSE=429.6987, MAE=161.5488, MRE=165.32%

=== ALA throughput split80 | run 4/5 | seed=3042 ===
[ALA throughput split80/20] R2=0.4832, RMSE=438.7246, MAE=160.5370, MRE=167.82% (train=107817, test=26955)
R2=0.4832, RMSE=438.7246, MAE=160.5370, MRE=167.82%

=== ALA throughput split80 | run 5/5 | seed=4042 ===
[ALA throughput split80/20] R2=0.4908, RMSE=423.7486, MAE=159.4629, MRE=170.59% (train=107817, t